<a href="https://colab.research.google.com/github/RamSankarS/ML-GA-Motif-Discovery/blob/main/ace2motif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython scikit-learn xgboost seaborn tqdm datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.3 MB/s eta 0:00:00


In [ ]:
from google.colab import files
import zipfile

uploaded_zip = files.upload()

zip_filename = 'covid19-ace2-variants.zip'
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('./covid19-ace2-variants.zip')

print("Folder extracted successfully.")


In [ ]:
import os
repo_dir = "covid19-ace2-variants/covid19-ace2-variants"
for root, dirs, files in os.walk(repo_dir):
    level = root.replace(repo_dir, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

covid19-ace2-variants/
    LICENSE
    snp_table.csv
    ACE2_CDS.fa
    CITATION.bib
    proteofav.ipynb
    directory_structure.txt
    covid_project_utils.py
    README.md
    ACE2-variants-structure-and-assays.ipynb
    ACE2_HUMAN.fa
    codons.ipynb
    .git/
        packed-refs
        HEAD
        index
        config
        description
        refs/
            remotes/
                origin/
                    HEAD
            tags/
            heads/
                master
        info/
            exclude
        branches/
        logs/
            HEAD
            refs/
                remotes/
                    origin/
                        HEAD
                heads/
                    master
        objects/
            pack/
                pack-f57f803a3623f1cafb635f04f3d1fd24c0bb2de5.pack
                pack-f57f803a3623f1cafb635f04f3d1fd24c0bb2de5.idx
                pack-f57f803a3623f1cafb635f04f3d1fd24c0bb2de5.rev
            info/
        hooks/
         

In [ ]:
import pandas as pd

# Adjust the path if needed.
snp_table_path = os.path.join(repo_dir, "snp_table.csv")
snp_df = pd.read_csv(snp_table_path)
print("First few rows of snp_table.csv:")
display(snp_df.head())


First few rows of snp_table.csv:


,from_codon,to_codon,from_aa,to_aa,n_subs
0,TTT,TTC,F,F,1
1,TTT,TTA,F,L,1
2,TTT,TTG,F,L,1
3,TTT,TCT,F,S,1
4,TTT,TAT,F,Y,1


In [ ]:
from Bio import SeqIO

# Load ACE2_CDS.fa
cds_fasta_path = os.path.join(repo_dir, "ACE2_CDS.fa")
cds_records = list(SeqIO.parse(cds_fasta_path, "fasta"))
print(f"Found {len(cds_records)} record(s) in ACE2_CDS.fa.")
print("First record:")
print(cds_records[0].format("fasta"))

# Load ACE2_HUMAN.fa
human_fasta_path = os.path.join(repo_dir, "ACE2_HUMAN.fa")
human_records = list(SeqIO.parse(human_fasta_path, "fasta"))
print(f"\nFound {len(human_records)} record(s) in ACE2_HUMAN.fa.")
print("First record:")
print(human_records[0].format("fasta"))

Found 1 record(s) in ACE2_CDS.fa.
First record:
>CDS|ENST00000427411/1-2415
ATGTCAAGCTCTTCCTGGCTCCTTCTCAGCCTTGTTGCTGTAACTGCTGCTCAGTCCACC
ATTGAGGAACAGGCCAAGACATTTTTGGACAAGTTTAACCACGAAGCCGAAGACCTGTTC
TATCAAAGTTCACTTGCTTCTTGGAATTATAACACCAATATTACTGAAGAGAATGTCCAA
AACATGAATAATGCTGGGGACAAATGGTCTGCCTTTTTAAAGGAACAGTCCACACTTGCC
CAAATGTATCCACTACAAGAAATTCAGAATCTCACAGTCAAGCTTCAGCTGCAGGCTCTT
CAGCAAAATGGGTCTTCAGTGCTCTCAGAAGACAAGAGCAAACGGTTGAACACAATTCTA
AATACAATGAGCACCATCTACAGTACTGGAAAAGTTTGTAACCCAGATAATCCACAAGAA
TGCTTATTACTTGAACCAGGTTTGAATGAAATAATGGCAAACAGTTTAGACTACAATGAG
AGGCTCTGGGCTTGGGAAAGCTGGAGATCTGAGGTCGGCAAGCAGCTGAGGCCATTATAT
GAAGAGTATGTGGTCTTGAAAAATGAGATGGCAAGAGCAAATCATTATGAGGACTATGGG
GATTATTGGAGAGGAGACTATGAAGTAAATGGGGTAGATGGCTATGACTACAGCCGCGGC
CAGTTGATTGAAGATGTGGAACATACCTTTGAAGAGATTAAACCATTATATGAACATCTT
CATGCCTATGTGAGGGCAAAGTTGATGAATGCCTATCCTTCCTATATCAGTCCAATTGGA
TGCCTCCCTGCTCATTTGCTTGGTGATATGTGGGGTAGATTTTGGACAAATCTGTACTCT
TTGACAGTTCCCTTTGGACAGAAACCAAACATAGATGTTACTGATGCAATGGTGGACCAG
GCCTGGGAT

In [ ]:
import glob
import os
import pandas as pd

# Construct the search pattern to find all MutaBind2.csv files recursively
mutabind_pattern = os.path.join(repo_dir, "MutaBind2", "**", "MutaBind2.csv")
mutabind_csv_files = glob.glob(mutabind_pattern, recursive=True)
print("Found MutaBind2 CSV files:")
print(mutabind_csv_files)

# Read and combine all MutaBind2.csv files, skipping the first row in each file.
mutabind_dfs = []
for file in mutabind_csv_files:
    # Skip the first row which contains the title.
    df = pd.read_csv(file, skiprows=1)
    # Extract the parent folder name as a category identifier.
    parent_folder = os.path.basename(os.path.dirname(file))
    df["category"] = parent_folder
    mutabind_dfs.append(df)

if mutabind_dfs:
    combined_mutabind = pd.concat(mutabind_dfs, ignore_index=True)
    print("Combined MutaBind2 predictions (first 5 rows):")
    display(combined_mutabind.head())
else:
    print("No MutaBind2 CSV files found.")


Found MutaBind2 CSV files:
['covid19-ace2-variants/covid19-ace2-variants/MutaBind2/gnomAD-interface/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/Bat/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/gnomAD/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/Pig/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/alanine-scan/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/Civet/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/Mouse/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/R439N-model/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/R439N-model/gnomAD-interface/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/R439N-model/gnomAD-extended-interface/MutaBind2.csv', 'covid19-ace2-variants/covid19-ace2-variants/MutaBind2/R439N-model/ortholog-single-submission/MutaBind2.csv', 'covid19-ace2-va

,Mutated Chain,Mutation,DDG,Interface?,Deleterious?,DDE_vdw,DDG_solv,DDG_fold,CS,SA_com_wt,SA_part_wt,N_cont_wt,bootstrapFC,Model_Bias,category,Mutation ID,dE_vdw_wt
0,A,S19P,0.08,yes,no,-0.1241,-0.1197,-0.1586,0.2505,-0.2804,0.0103,0.0757,-0.0018,0.4286,gnomAD-interface,NaN,NaN
1,A,T27A,0.64,yes,no,-0.0347,-0.0829,-0.1390,0.0341,0.2648,0.1305,0.0359,-0.0018,0.4286,gnomAD-interface,NaN,NaN
2,A,E35K,0.38,yes,no,-0.2286,0.0473,-0.1360,0.3559,-0.1565,-0.0098,0.0791,-0.0018,0.4286,gnomAD-interface,NaN,NaN
3,A,E37K,0.96,yes,no,-0.1633,0.7271,0.0088,0.2456,0.0008,-0.2978,0.0113,-0.0018,0.4286,gnomAD-interface,NaN,NaN
4,A,M82I,0.23,yes,no,-0.1547,-0.1048,0.0426,0.1563,-0.2867,0.0720,0.0774,-0.0018,0.4286,gnomAD-interface,NaN,NaN


In [ ]:
from Bio import SeqIO
from Bio.Data import CodonTable
import pandas as pd

# 1. Load the ACE2 coding sequence
ace2_cds_path = "ACE2_CDS.fa"  # Adjust if needed
ace2_record = list(SeqIO.parse(ace2_cds_path, "fasta"))[0]  # assume only one record in the FASTA
ace2_seq = str(ace2_record.seq).upper()

# 2. Split into codons
codons = [ace2_seq[i : i+3] for i in range(0, len(ace2_seq), 3)]

# 3. Translate each codon to get the reference amino acid
standard_table = CodonTable.unambiguous_dna_by_id[1]  # Standard table, ID=1

def translate_codon(codon):
    """Translate a 3-nt codon into an amino acid (single letter).
       Returns '*' for stop codons or 'X' if not found."""
    # Some codons might be ambiguous; handle with get().
    return standard_table.forward_table.get(codon,
           standard_table.stop_codons and '*')

# Build a reference DataFrame mapping each codon index to (codon, amino_acid)
codon_df = pd.DataFrame({
    "codon_index": range(1, len(codons) + 1),
    "ref_codon": codons
})
codon_df["ref_aa"] = codon_df["ref_codon"].apply(translate_codon)

print("Reference codon table (first 10 rows):")
display(codon_df.head(10))


Reference codon table (first 10 rows):


,codon_index,ref_codon,ref_aa
0,1,ATG,M
1,2,TCA,S
2,3,AGC,S
3,4,TCT,S
4,5,TCC,S
5,6,TGG,W
6,7,CTC,L
7,8,CTT,L
8,9,CTC,L
9,10,AGC,S


In [ ]:
# Determine which columns correspond to mutations.
# Here, we assume 'WT' and 'source_file' are non-mutation columns.
assay_cols = [col for col in combined_assays.columns if col.strip() not in ['WT', 'source_file']]
print("Assay mutation columns detected:", assay_cols)

# Pivot the experimental assay data to long format:
assay_long = combined_assays.melt(id_vars=['WT', 'source_file'],
                                  value_vars=assay_cols,
                                  var_name='mutation',
                                  value_name='binding_affinity')

print("Experimental assay data in long format:")
display(assay_long.head())


Assay mutation columns detected: [' 97', 'P84S', ' 82']
Experimental assay data in long format:


,WT,source_file,mutation,binding_affinity
0,K31D,mutant-S1-Ig-association-from-Fig3.csv,97,2.0
1,E37A,mutant-S1-Ig-association-from-Fig3.csv,97,94.0
2,D38A,mutant-S1-Ig-association-from-Fig3.csv,97,90.0
3,Y41A,mutant-S1-Ig-association-from-Fig3.csv,97,12.0
4,K68D,mutant-S1-Ig-association-from-Fig3.csv,97,72.0


In [ ]:
snp_df = snp_df.rename(columns={
    "from_codon": "snp_from_codon",
    "to_codon":   "snp_to_codon",
    "from_aa":    "snp_from_aa",
    "to_aa":      "snp_to_aa"
})
print("SNP table:")
display(snp_df.head(10))


SNP table:


,snp_from_codon,snp_to_codon,snp_from_aa,snp_to_aa,n_subs,mutation,mutation_base
0,TTT,TTC,F,F,1,F1F,F1
1,TTT,TTA,F,L,1,F1L,F1
2,TTT,TTG,F,L,1,F1L,F1
3,TTT,TCT,F,S,1,F1S,F1
4,TTT,TAT,F,Y,1,F1Y,F1
5,TTT,TGT,F,C,1,F1C,F1
6,TTT,CTT,F,L,1,F1L,F1
7,TTT,ATT,F,I,1,F1I,F1
8,TTT,GTT,F,V,1,F1V,F1
9,TTC,TTT,F,F,1,F1F,F1


In [ ]:
def find_positions_for_snp(row, codon_df):
    """
    Given a row with (snp_from_codon, snp_from_aa, snp_to_codon, snp_to_aa),
    return all possible codon_index positions in codon_df that match the 'from' codon/AA.
    Also verify that to_codon translates to to_aa for consistency.
    """
    from_codon = row["snp_from_codon"]
    from_aa    = row["snp_from_aa"]
    to_codon   = row["snp_to_codon"]
    to_aa      = row["snp_to_aa"]

    # Check that to_codon -> to_aa is consistent
    # If it's not, we might as well return an empty list
    if translate_codon(to_codon) != to_aa:
        return []

    # Filter codon_df where reference codon == from_codon AND reference AA == from_aa
    matches = codon_df[
        (codon_df["ref_codon"] == from_codon) &
        (codon_df["ref_aa"] == from_aa)
    ]

    # Return the list of matching indices
    return list(matches["codon_index"])

# Apply this to each row in snp_df
all_positions = []
for i, row in snp_df.iterrows():
    positions = find_positions_for_snp(row, codon_df)
    # If no positions, we skip or note it
    # If multiple positions, we create multiple rows
    if len(positions) == 0:
        # Could store a placeholder or skip
        all_positions.append({
            **row.to_dict(),
            "codon_index": None  # no match
        })
    else:
        # For each matching position, create a row
        for pos in positions:
            new_row = {**row.to_dict(), "codon_index": pos}
            all_positions.append(new_row)

# Create a new DataFrame with codon_index appended
snp_positions_df = pd.DataFrame(all_positions)
print("SNP table with potential codon positions:")
display(snp_positions_df.head(20))


SNP table with potential codon positions:


,snp_from_codon,snp_to_codon,snp_from_aa,snp_to_aa,n_subs,mutation,mutation_base,codon_index
0,TTT,TTC,F,F,1,F1F,F1,28.0
1,TTT,TTC,F,F,1,F1F,F1,32.0
2,TTT,TTC,F,F,1,F1F,F1,72.0
3,TTT,TTC,F,F,1,F1F,F1,230.0
4,TTT,TTC,F,F,1,F1F,F1,274.0
5,TTT,TTC,F,F,1,F1F,F1,285.0
6,TTT,TTC,F,F,1,F1F,F1,315.0
7,TTT,TTC,F,F,1,F1F,F1,390.0
8,TTT,TTC,F,F,1,F1F,F1,428.0
9,TTT,TTC,F,F,1,F1F,F1,452.0


In [ ]:
def make_protein_mutation_label(row):
    # e.g. from_aa = 'F', codon_index = 84, to_aa = 'S' -> "F84S"
    if pd.isna(row["codon_index"]):
        return None
    return f"{row['snp_from_aa']}{row['codon_index']}{row['snp_to_aa']}"

snp_positions_df["mutation"] = snp_positions_df.apply(make_protein_mutation_label, axis=1)

print("SNP table with standard protein mutation labels:")
display(snp_positions_df.head(20))


SNP table with standard protein mutation labels:


,snp_from_codon,snp_to_codon,snp_from_aa,snp_to_aa,n_subs,mutation,mutation_base,codon_index
0,TTT,TTC,F,F,1,F28.0F,F1,28.0
1,TTT,TTC,F,F,1,F32.0F,F1,32.0
2,TTT,TTC,F,F,1,F72.0F,F1,72.0
3,TTT,TTC,F,F,1,F230.0F,F1,230.0
4,TTT,TTC,F,F,1,F274.0F,F1,274.0
5,TTT,TTC,F,F,1,F285.0F,F1,285.0
6,TTT,TTC,F,F,1,F315.0F,F1,315.0
7,TTT,TTC,F,F,1,F390.0F,F1,390.0
8,TTT,TTC,F,F,1,F428.0F,F1,428.0
9,TTT,TTC,F,F,1,F452.0F,F1,452.0


In [ ]:
non_synonymous = snp_positions_df[snp_positions_df["snp_from_aa"] != snp_positions_df["snp_to_aa"]]

print(f"Total non-synonymous mutations found: {len(non_synonymous)}")
display(non_synonymous.head(20))


Total non-synonymous mutations found: 5401


,snp_from_codon,snp_to_codon,snp_from_aa,snp_to_aa,n_subs,mutation,mutation_base,codon_index
22,TTT,TTA,F,L,1,F28.0L,F1,28.0
23,TTT,TTA,F,L,1,F32.0L,F1,32.0
24,TTT,TTA,F,L,1,F72.0L,F1,72.0
25,TTT,TTA,F,L,1,F230.0L,F1,230.0
26,TTT,TTA,F,L,1,F274.0L,F1,274.0
27,TTT,TTA,F,L,1,F285.0L,F1,285.0
28,TTT,TTA,F,L,1,F315.0L,F1,315.0
29,TTT,TTA,F,L,1,F390.0L,F1,390.0
30,TTT,TTA,F,L,1,F428.0L,F1,428.0
31,TTT,TTA,F,L,1,F452.0L,F1,452.0


In [ ]:
# Create a mutation_base column in the mCSM-PPI2 predictions DataFrame
combined_mcsm['mutation_base'] = combined_mcsm['wild-type'] + combined_mcsm['res-number'].astype(str)

print("mCSM-PPI2 predictions after adding mutation_base:")
display(combined_mcsm.head())
combined_mcsm


mCSM-PPI2 predictions after adding mutation_base:


,chain,wild-type,res-number,distance-to-interface,source_file,mutation_base
0,A,SER,19,2.631,mCSM-PPI2-server-interface.csv,SER19
1,A,GLN,24,3.031,mCSM-PPI2-server-interface.csv,GLN24
2,A,THR,27,3.703,mCSM-PPI2-server-interface.csv,THR27
3,A,PHE,28,3.376,mCSM-PPI2-server-interface.csv,PHE28
4,A,ASP,30,4.116,mCSM-PPI2-server-interface.csv,ASP30


,chain,wild-type,res-number,distance-to-interface,source_file,mutation_base
0,A,SER,19,2.631,mCSM-PPI2-server-interface.csv,SER19
1,A,GLN,24,3.031,mCSM-PPI2-server-interface.csv,GLN24
2,A,THR,27,3.703,mCSM-PPI2-server-interface.csv,THR27
3,A,PHE,28,3.376,mCSM-PPI2-server-interface.csv,PHE28
4,A,ASP,30,4.116,mCSM-PPI2-server-interface.csv,ASP30
5,A,LYS,31,2.925,mCSM-PPI2-server-interface.csv,LYS31
6,A,HIS,34,3.049,mCSM-PPI2-server-interface.csv,HIS34
7,A,GLU,35,2.933,mCSM-PPI2-server-interface.csv,GLU35
8,A,GLU,37,3.185,mCSM-PPI2-server-interface.csv,GLU37
9,A,ASP,38,2.991,mCSM-PPI2-server-interface.csv,ASP38


In [ ]:
from Bio import SeqUtils

# Convert single-letter amino acid codes to three-letter codes
def convert_aa(single_letter):
    try:
        return SeqUtils.IUPACData.protein_letters_1to3[single_letter.upper()].upper()
    except KeyError:
        return None  # Handle unexpected characters

# Apply conversion to snp_positions_df to match mCSM-PPI2 format
snp_positions_df["wild-type"] = snp_positions_df["snp_from_aa"].apply(convert_aa)
snp_positions_df["mutation_base"] = snp_positions_df["wild-type"] + snp_positions_df["codon_index"].fillna(0).astype(int).astype(str)

# Merge with mCSM-PPI2 predictions
df_merged = snp_positions_df.merge(combined_mcsm, on="mutation_base", how="left")

# Display summary
print(f"Total SNP mutations matched with structural predictions: {df_merged['mutation_base'].notna().sum()} out of {len(snp_positions_df)}")
display(df_merged.head(20))


Total SNP mutations matched with structural predictions: 6925 out of 6925


,snp_from_codon,snp_to_codon,snp_from_aa,snp_to_aa,n_subs,mutation,mutation_base,codon_index,wild-type_x,chain,wild-type_y,res-number,distance-to-interface,source_file
0,TTT,TTC,F,F,1,F28.0F,PHE28,28.0,PHE,A,PHE,28.0,3.376,mCSM-PPI2-server-interface.csv
1,TTT,TTC,F,F,1,F32.0F,PHE32,32.0,PHE,NaN,NaN,NaN,NaN,NaN
2,TTT,TTC,F,F,1,F72.0F,PHE72,72.0,PHE,NaN,NaN,NaN,NaN,NaN
3,TTT,TTC,F,F,1,F230.0F,PHE230,230.0,PHE,NaN,NaN,NaN,NaN,NaN
4,TTT,TTC,F,F,1,F274.0F,PHE274,274.0,PHE,NaN,NaN,NaN,NaN,NaN
5,TTT,TTC,F,F,1,F285.0F,PHE285,285.0,PHE,NaN,NaN,NaN,NaN,NaN
6,TTT,TTC,F,F,1,F315.0F,PHE315,315.0,PHE,NaN,NaN,NaN,NaN,NaN
7,TTT,TTC,F,F,1,F390.0F,PHE390,390.0,PHE,NaN,NaN,NaN,NaN,NaN
8,TTT,TTC,F,F,1,F428.0F,PHE428,428.0,PHE,NaN,NaN,NaN,NaN,NaN
9,TTT,TTC,F,F,1,F452.0F,PHE452,452.0,PHE,NaN,NaN,NaN,NaN,NaN
